# S1 — Pipeline complet : résistance du béton (génie civil)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ic1/notebooks/IC1_S1_pipeline_beton.ipynb)
> Données : dossier `IC1_TP_donnees`. Bibliothèques : pandas, numpy, matplotlib, scikit-learn.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
# Elle récupère les données et les tests de vérification : depuis le dépôt
# (../data, ../tests) si vous l'avez téléchargé, sinon depuis GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ic1/'
FICHIERS = [('data/beton_8_constituants.csv', 'beton_8_constituants.csv'), ('tests/s1/q1.py', 'tests/q1.py'), ('tests/s1/q2.py', 'tests/q2.py'), ('tests/s1/q3.py', 'tests/q3.py'), ('tests/s1/q4.py', 'tests/q4.py')]
os.makedirs('tests', exist_ok=True)
for source, cible in FICHIERS:
    if os.path.exists(cible):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, cible)                      # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + source, cible)  # Colab : téléchargement
print('Prêt :', ', '.join(c for _, c in FICHIERS))

In [ ]:
# Vérification automatique des réponses (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, cross_val_score, cross_val_predict, learning_curve, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, ConfusionMatrixDisplay
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
b = pd.read_csv('beton_8_constituants.csv'); print(b.isna().sum())
X, y = b.drop(columns='resistance_MPa'), b['resistance_MPa']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

## 1. Trois modèles dans des pipelines (aucune fuite)

In [ ]:
modeles = {
 'lin': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LinearRegression()),
 'ridge': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=10)),
 'rf': make_pipeline(SimpleImputer(strategy='median'), RandomForestRegressor(400, min_samples_leaf=2, random_state=0))}
for nom, m in modeles.items():
    m.fit(X_tr, y_tr); p = m.predict(X_te)
    print(nom, round(mean_absolute_error(y_te, p), 2), round(r2_score(y_te, p), 3))

## 2. Courbe d’apprentissage : faut-il plus d’essais ?

In [ ]:
sz, tr, va = learning_curve(modeles['rf'], X, y, cv=5, scoring='r2', train_sizes=np.linspace(.1, 1, 6))
plt.plot(sz, tr.mean(1), 'o-', label='entraînement'); plt.plot(sz, va.mean(1), 'o-', label='validation'); plt.legend(); plt.show()

## Exercices vérifiés automatiquement
Remplacez chaque `...` par votre réponse, exécutez la cellule : la vérification indique tout de suite si c'est correct.
Ces tests servent à vous entraîner ; la note du TP est attribuée sur le notebook rendu.

**Q1.** Quel modèle a la plus petite MAE sur le jeu de test ? Stockez son nom (`'lin'`, `'ridge'` ou `'rf'`) dans `meilleur_modele`.

In [ ]:
meilleur_modele = ...
grader.check("q1")

**Q2.** Calculez le R² de la forêt (`modeles['rf']`) sur le jeu de test et stockez-le dans `r2_rf`.

In [ ]:
r2_rf = ...
grader.check("q2")

**Q3.** Avec la courbe d'apprentissage (`tr`, `va`), calculez l'écart entre le score d'entraînement et le score de validation **à la plus grande taille** d'échantillon. Stockez-le dans `ecart_final`.
 Un grand écart indique-t-il plutôt un biais ou une variance ?

In [ ]:
ecart_final = ...
grader.check("q3")

**Q4 (bonus).** Le superplastifiant manquant signifie peut-être « pas de superplastifiant ». Refaites la forêt en imputant par **0** (`SimpleImputer(strategy='constant', fill_value=0)`, mêmes paramètres de forêt), et stockez sa MAE sur le test dans `mae_rf_zero`.
 Quelle hypothèse vous paraît la plus réaliste ?

In [ ]:
mae_rf_zero = ...
grader.check("q4")

Pour tout vérifier d'un coup :

In [ ]:
grader.check_all()